In [1]:
# Financial Analysis Chatbot
# Author: Manan Mehta

import os
import pandas as pd
from datetime import datetime

# Load the CSV produced by the extraction notebook
# Looks for the CSV next to this notebook first, then in the output folder
CSV_PATH = "combined_financial_analysis.csv"
if not os.path.exists(CSV_PATH):
    CSV_PATH = os.path.join("combined_analysis_output", "combined_financial_analysis.csv")
df = pd.read_csv(CSV_PATH)

METRICS = {
    "Total Revenue": ["total revenue", "revenue", "sales"],
    "Net Income": ["net income", "profit", "earnings"],
    "Total Assets": ["total assets", "assets"],
    "Total Liabilities": ["total liabilities", "liabilities", "debt"],
    "Cash Flow from Operating Activities": ["cash flow", "operating cash", "cash from operations"],
}

INTENTS = {
    "company_list": ["which companies", "list companies", "available companies", "list of companies"],
    "comparison": ["compare", "vs", "versus"],
}

In [2]:
class FinancialChatbot:
    def __init__(self, data):
        self.df = data
        self.user = "Manan Mehta"
        self.companies = list(self.df['Company'].unique())

    @staticmethod
    def format_currency(amount):
        """Values are in USD millions"""
        return f"${amount:,.0f}M"

    def detect_companies(self, text):
        found = [c for c in self.companies if c.lower() in text]
        return found or self.companies

    def detect_metric(self, text):
        for metric, keywords in METRICS.items():
            if any(k in text for k in keywords):
                return metric
        return None

    def latest(self, company, metric):
        d = self.df[self.df['Company'] == company].sort_values('Year')
        row = d.iloc[-1]
        yoy = None
        if len(d) >= 2 and d.iloc[-2][metric] != 0:
            yoy = (row[metric] - d.iloc[-2][metric]) / d.iloc[-2][metric] * 100
        return int(row['Year']), row[metric], yoy

    def process_query(self, user_input):
        text = user_input.lower().strip()
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        header = f"[{now}] Response for {self.user}:\n"

        if any(p in text for p in INTENTS["company_list"]):
            return header + "Available companies: " + ", ".join(self.companies)

        metric = self.detect_metric(text)
        if metric is None:
            return header + (
                "I can answer questions on: Total Revenue, Net Income, Total Assets, "
                "Total Liabilities and Operating Cash Flow for " + ", ".join(self.companies) +
                ". Try 'Apple net income' or 'compare revenue Microsoft vs Tesla'."
            )

        companies = self.detect_companies(text)
        lines = [f"{metric} (latest year):"]
        results = []
        for c in companies:
            year, value, yoy = self.latest(c, metric)
            results.append((c, value))
            line = f"  {c} ({year}): {self.format_currency(value)}"
            if yoy is not None:
                line += f" | YoY change: {yoy:.1f}%"
            lines.append(line)

        if any(p in text for p in INTENTS["comparison"]) and len(results) > 1:
            top = max(results, key=lambda x: x[1])
            lines.append(f"Highest: {top[0]}")

        return header + "\n".join(lines)

In [3]:
chatbot = FinancialChatbot(df)

def ask_chatbot(query):
    return chatbot.process_query(query)

# Demo queries
for q in ["Which companies do you have data for?",
          "Microsoft total revenue",
          "Compare net income Apple vs Tesla",
          "What are the total liabilities?",
          "Operating cash flow"]:
    print("Q:", q)
    print(ask_chatbot(q), "\n")

Q: Which companies do you have data for?
[2026-09-22 15:43:37] Response for Manan Mehta:
Available companies: Apple, Microsoft, Tesla 

Q: Microsoft total revenue
[2026-09-22 15:43:37] Response for Manan Mehta:
Total Revenue (latest year):
  Microsoft (2024): $211,915M | YoY change: 6.9% 

Q: Compare net income Apple vs Tesla
[2026-09-22 15:43:37] Response for Manan Mehta:
Net Income (latest year):
  Apple (2024): $96,995M | YoY change: -2.8%
  Tesla (2024): $10,000M | YoY change: 11.1%
Highest: Apple 

Q: What are the total liabilities?
[2026-09-22 15:43:37] Response for Manan Mehta:
Total Liabilities (latest year):
  Apple (2024): $287,912M | YoY change: 10.3%
  Microsoft (2024): $198,193M | YoY change: 16.7%
  Tesla (2024): $100,000M | YoY change: 11.1% 

Q: Operating cash flow
[2026-09-22 15:43:37] Response for Manan Mehta:
Cash Flow from Operating Activities (latest year):
  Apple (2024): $110,543M | YoY change: -9.5%
  Microsoft (2024): $87,708M | YoY change: 0.6%
  Tesla (2024):

In [4]:
# Interactive mode: set INTERACTIVE = True and run this cell
INTERACTIVE = False

if INTERACTIVE:
    print("Financial Analysis Chatbot (type 'exit' to stop)")
    while True:
        query = input("Your question: ")
        if query.lower().strip() == "exit":
            break
        print("\n" + ask_chatbot(query) + "\n")